# 11 — TROPOMI XCH₄ observation-quality and representativeness audit (2019–2025)

**Study:** Hakaluki Haor | exploratory atmospheric methane observations alongside Sentinel-1 low-VV screening and hydroclimate.

**This notebook deliberately does not estimate local methane emissions.** XCH₄ is column-averaged dry-air methane mixing ratio in ppb, not wetland surface methane flux. The available monthly Level-3 Earth Engine summaries do **not** permit native Level-2 footprint-level QA, uncertainty evaluation, wind attribution, or verification that retrievals are specifically observing methane emitted over the haor.

**Research goals:** audit all 84 months, expose missingness and seasonal sampling bias, examine coverage sensitivity, and identify which follow-up analyses are feasible. No interpolation of missing XCH₄ and no artificial increase in sample size by regridding.

**Data:** `data/processed/hakaluki_monthly_2019_2025.csv` (previous processing) and `results/tables/inundation09_monthly_2019_2025.csv` (QA-v2/v3 low-VV time series, fixed ascending orbit 41).

**Data references:** [Earth Engine Sentinel-5P OFFL L3 CH₄](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_OFFL_L3_CH4) · [TROPOMI methane product](https://www.tropomi.eu/data-products/methane). Earth Engine L3 is built from L2 data filtered upstream; native sounding QA and L2 averaging kernels are **not** preserved in the monthly CSV. The Earth Engine dataset documents residual artifacts and an outage 26 July–31 August 2022.

## 11A — Load the real monthly data and verify the 84-month calendar

Fail on ambiguous fields or dates rather than guessing. Earth Engine image counts represent *L3 orbit tiles*, **not** independent native methane soundings.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r"F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project")
SOURCE = ROOT / "data/processed/hakaluki_monthly_2019_2025.csv"
RADAR = ROOT / "results/tables/inundation09_monthly_2019_2025.csv"
TABLES, FIGURES, REPORTS = (ROOT / "results/tables", ROOT / "results/figures", ROOT / "results/reports")
for folder in (TABLES, FIGURES, REPORTS): folder.mkdir(parents=True, exist_ok=True)
for p in [SOURCE, RADAR]:
    if not p.is_file(): raise FileNotFoundError(p)

x = pd.read_csv(SOURCE)
r = pd.read_csv(RADAR)
print("Methane source columns:", x.columns.tolist())
print("Radar source columns:", r.columns.tolist())
CH4 = 'ch4_xch4_ppb_roi'
RING = 'ch4_xch4_ppb_ring'
DIFF = 'ch4_roi_minus_ring_ppb'
COV = 'ch4_l3_grid_coverage_fraction'
TILES = 'ch4_images_tiled'
REQUIRED = [CH4, RING, DIFF, COV, TILES]
missing = [c for c in REQUIRED if c not in x.columns]
if missing: raise KeyError(f"Missing methane fields {missing}. Inspect the source CSV rather than guessing units or columns.")

def monthly_date(frame, name):
    if 'period' in frame.columns:
        series = frame['period'].astype(str).str[:7]
        date = pd.to_datetime(series + '-01', errors='raise')
    elif 'date' in frame.columns:
        date = pd.to_datetime(frame['date'], errors='raise').dt.to_period('M').dt.to_timestamp()
    elif {'year', 'month'}.issubset(frame.columns):
        date = pd.to_datetime(dict(year=frame['year'], month=frame['month'], day=1), errors='raise')
    else:
        raise KeyError(f"{name}: no period/date or year+month fields")
    if date.isna().any(): raise ValueError(f"{name}: unparseable monthly date")
    return date

x['date'] = monthly_date(x, 'methane')
r['date'] = monthly_date(r, 'radar')
expected = pd.date_range('2019-01-01', '2025-12-01', freq='MS')
for frame,name in [(x,'methane'),(r,'radar')]:
    if frame.date.duplicated().any(): raise ValueError(f"{name}: duplicated months")
    if len(frame) != 84 or set(frame.date) != set(expected):
        raise ValueError(f"{name}: monthly calendar does not exactly match 2019–2025")

for c in REQUIRED:
    x[c] = pd.to_numeric(x[c], errors='coerce')
if x[[COV,TILES]].isna().any().any():
    raise ValueError('Missing coverage or L3 image-tile count; review original processing.')
if not x[COV].between(0, 1.001).all(): raise ValueError('L3 grid coverage out of range')
if (x[TILES]<0).any(): raise ValueError('Negative L3 image-tile count')
x[COV] = x[COV].clip(0, 1)

radar_cols = ['date', 'nominal_lowvv_km2', 'lowvv_m19_km2', 'lowvv_m15_km2',
              'scene_count', 'comparison_eligible', 'config_hash']
missing_radar = [c for c in radar_cols if c not in r.columns]
if missing_radar: raise KeyError(f"Missing radar fields: {missing_radar}")

m = x.merge(r[radar_cols], on='date', validate='one_to_one', how='inner', suffixes=('','_radar'))
m = m.sort_values('date').reset_index(drop=True)
m['year'] = m.date.dt.year
m['month'] = m.date.dt.month
m['xch4_available'] = m[CH4].notna()
m['ring_available'] = m[RING].notna()
m['difference_available'] = m[DIFF].notna()
m['is_monsoon_may_oct'] = m.month.between(5,10)
m['l3_grid_support'] = m[COV].where(m.xch4_available, 0.0)

# Reject strong internal arithmetic inconsistency; tiny rounding is acceptable.
both = m[[CH4,RING,DIFF]].notna().all(axis=1)
if both.any():
    mismatch = (m.loc[both, CH4] - m.loc[both, RING] - m.loc[both, DIFF]).abs().max()
    print('Max ROI - ring arithmetic discrepancy (ppb):', round(float(mismatch),7))
    if mismatch > 0.1:
        warnings.warn('ROI–ring differences do not match; inspect original background method.')

print('=== 11A INPUT QA ===')
print('Calendar months:',len(m))
print('ROI XCH4 available:',int(m.xch4_available.sum()))
print('ROI-minus-ring available:',int(m.difference_available.sum()))
print('L3 collection tiles/month is NOT number of native soundings.')
print('Radar configuration fingerprint:', m.config_hash.dropna().unique().tolist())
display(m[['date', CH4, RING, DIFF, COV, TILES, 'nominal_lowvv_km2']].head(12))

## 11B — Missingness, retrieval support and seasonal representativeness

A month with nominal L3 coverage is **not** equivalent to a month with independently validated Level-2 methane soundings. We record L3 spatial grid support as a diagnostic proxy only.

In [ ]:
annual = (m.groupby('year').agg(
    months=('date','size'), roi_months=('xch4_available','sum'),
    difference_months=('difference_available','sum'),
    median_grid_support_when_available=(COV,lambda s: float(s[m.loc[s.index,'xch4_available']].median()) if m.loc[s.index,'xch4_available'].any() else np.nan),
    mean_l3_tiles=(TILES,'mean')).reset_index())
annual['roi_availability_percent'] = 100*annual.roi_months/annual.months
seasonal = (m.groupby('month').agg(n_years=('date','size'), roi_months=('xch4_available','sum'),
                                  difference_months=('difference_available','sum'),
                                  median_l3_grid_support=(COV,'median')).reset_index())
seasonal['roi_availability_percent'] = 100*seasonal.roi_months/seasonal.n_years
season_groups = (m.groupby('is_monsoon_may_oct').agg(
    months=('date','size'), roi_xch4_months=('xch4_available','sum'),
    roi_minus_ring_months=('difference_available','sum')).reset_index())
season_groups['availability_percent'] = 100*season_groups.roi_xch4_months/season_groups.months
season_groups['season'] = np.where(season_groups.is_monsoon_may_oct, 'May–October', 'November–April')

print('=== 11B ANNUAL AVAILABILITY ==='); display(annual)
print('=== 11B CALENDAR-MONTH AVAILABILITY ==='); display(seasonal)
print('=== 11B MONSOON AVAILABILITY ==='); display(season_groups[['season','months','roi_xch4_months','roi_minus_ring_months','availability_percent']])
print('=== MISSING 2022 MONTHS ===')
display(m.loc[(m.year==2022)&~m.xch4_available,['date',TILES,COV,'nominal_lowvv_km2']])

annual.to_csv(TABLES/'methane11_annual_availability.csv',index=False)
seasonal.to_csv(TABLES/'methane11_monthly_availability.csv',index=False)
season_groups.to_csv(TABLES/'methane11_seasonal_availability.csv',index=False)

## 11C — Availability and spatial-grid support figures

In [ ]:
arr = m.set_index(['year','month'])['xch4_available'].unstack('month').reindex(index=range(2019,2026), columns=range(1,13))
fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
axes[0].imshow(arr.astype(int), cmap='Greys', vmin=0, vmax=1, aspect='auto', interpolation='nearest')
axes[0].set_yticks(np.arange(7), labels=range(2019,2026)); axes[0].set_xticks(np.arange(12),labels=['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[0].set_title('Monthly ROI XCH₄ available (dark = present; white = missing)')
axes[0].set_ylabel('Year')
axes[1].plot(m.date, m[COV], lw=1,marker='.',markersize=3)
axes[1].scatter(m.loc[~m.xch4_available,'date'], np.zeros((~m.xch4_available).sum()), marker='x',color='black',label='Missing ROI XCH₄')
axes[1].set_ylim(-.06,1.06); axes[1].set_ylabel('L3 gridded support proxy');axes[1].set_xlabel('Month (UTC)')
axes[1].legend();axes[1].grid(alpha=.15)
fig.suptitle('Hakaluki TROPOMI XCH₄ sampling completeness | 2019–2025\nNot a native footprint-count or QA validation')
p=FIGURES/'methane11_availability_and_grid_support.png'; fig.savefig(p,dpi=220,bbox_inches='tight');plt.show();plt.close(fig)
print('Saved:',p)

## 11D — XCH₄ monthly series, study ROI and reference ring

Do not interpret apparent methane concentration trends, ROI–ring differences or peaks as local wetland emissions. The background ring is not an atmospheric-transport model.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13,8), sharex=True, constrained_layout=True)
axes[0].plot(m.date,m[CH4],'.-',lw=1,label='ROI XCH₄ (ppb)')
axes[0].plot(m.date,m[RING],'.-',lw=1,label='Reference-ring XCH₄ (ppb)')
axes[0].set_ylabel('Column XCH₄ (ppb)');axes[0].legend();axes[0].grid(alpha=.2)
axes[1].axhline(0,color='grey',lw=1)
mask=m[DIFF].notna()
axes[1].scatter(m.loc[mask,'date'],m.loc[mask,DIFF], c=m.loc[mask,COV],cmap='viridis',s=25,vmin=0,vmax=1)
axes[1].set_ylabel('ROI − ring XCH₄ (ppb)');axes[1].set_xlabel('Date');axes[1].grid(alpha=.2)
fig.suptitle('Hakaluki atmospheric methane column diagnostics (unvalidated sampling)')
p=FIGURES/'methane11_roi_ring_time_series.png';fig.savefig(p,dpi=230,bbox_inches='tight');plt.show();plt.close(fig)
print('Saved:',p)

## 11E — Are months missing preferentially under wet conditions?

Compare CH₄ availability against pre-existing low-VV screening. This is a *selection-bias diagnostic*, **not** a causal test or statistical significance test.

In [ ]:
from scipy.stats import spearmanr
q1,q3 = np.percentile(m.nominal_lowvv_km2.dropna(),[25,75])
lo = m.loc[m.nominal_lowvv_km2<=q1]
hi = m.loc[m.nominal_lowvv_km2>=q3]
strata = pd.DataFrame([
    {'stratum':'Low-VV bottom quartile','n_months':len(lo),'xch4_available_months':int(lo.xch4_available.sum()), 'availability_percent':100*lo.xch4_available.mean()},
    {'stratum':'Low-VV top quartile','n_months':len(hi),'xch4_available_months':int(hi.xch4_available.sum()), 'availability_percent':100*hi.xch4_available.mean()}])
print('=== 11E RADAR-DEPENDENT XCH4 AVAILABILITY ===');display(strata)
fig,ax=plt.subplots(figsize=(9,5))
ax.boxplot([m.loc[m.xch4_available,'nominal_lowvv_km2'].dropna(),m.loc[~m.xch4_available,'nominal_lowvv_km2'].dropna()],tick_labels=['ROI XCH₄ available','ROI XCH₄ missing'])
ax.set_ylabel('Monthly nominal low-VV area (km²)')
ax.set_title('Is methane observation availability conditional on wetland state?\nDescriptive diagnostic; SAR class unvalidated')
ax.grid(axis='y',alpha=.2)
p=FIGURES/'methane11_observation_selection_bias.png';fig.savefig(p,dpi=230,bbox_inches='tight');plt.show();plt.close(fig)
strata.to_csv(TABLES/'methane11_selection_bias_strata.csv',index=False)
print('Saved:',p)

## 11F — Coverage-sensitive exploratory relationships

**No imputation.** For each observed calendar month, subtract the calendar-month median computed from observed years. A minimum of three observed years in each calendar-month group is required. XCH₄ may show an unrelated multiyear atmospheric background trend, so even detrended/deseasonalized correlations cannot establish wetland methane flux or causal inundation response. Spatial grid support thresholds are explicitly *analytical sensitivity filters*, not official retrieval-quality thresholds.

In [ ]:
def calendar_month_departure(frame, variable, minimum_month_n=3):
    out = pd.Series(np.nan,index=frame.index,dtype='float64')
    for month,ids in frame.groupby('month').groups.items():
        val = pd.to_numeric(frame.loc[ids,variable],errors='coerce')
        if val.notna().sum()>=minimum_month_n:
            out.loc[ids] = val - val.median()
    return out

for variable in [CH4,DIFF,'nominal_lowvv_km2']:
    m[variable+'_seasonal_departure'] = calendar_month_departure(m,variable)
records=[]
for target in [CH4,DIFF]:
    for predictor in ['nominal_lowvv_km2']:
        for minimum_grid_support in [0.0,0.05,0.20]:
            subset=m.loc[(m[COV]>=minimum_grid_support)&m[target].notna(), [target+'_seasonal_departure',predictor+'_seasonal_departure']].dropna()
            rho = (float(spearmanr(subset.iloc[:,0],subset.iloc[:,1]).statistic) if len(subset)>=12 and subset.nunique().min()>1 else np.nan)
            records.append({'target':target,'predictor':predictor,'minimum_l3_grid_support':minimum_grid_support,
                'n_observed_months':len(subset),'spearman_rho_deseasonalized':rho,
                'validity':'exploratory; native L2 QA not checked; background trend and missingness unresolved'})
associations=pd.DataFrame(records)
print('=== 11F COVERAGE SENSITIVITY (NO HYPOTHESIS TESTS) ===');display(associations)
associations.to_csv(TABLES/'methane11_coverage_sensitive_associations.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax,target,title in zip(axes,[CH4,DIFF],['ROI XCH₄','ROI − ring XCH₄']):
    data=m.loc[m[target+'_seasonal_departure'].notna() & m.nominal_lowvv_km2_seasonal_departure.notna()]
    p=ax.scatter(data.nominal_lowvv_km2_seasonal_departure,data[target+'_seasonal_departure'],c=data[COV],vmin=0,vmax=1,cmap='viridis',s=32)
    ax.axhline(0,color='grey',lw=.7);ax.axvline(0,color='grey',lw=.7)
    ax.set_xlabel('Monthly low-VV departure (km²)');ax.set_ylabel('Month-centred XCH₄ departure (ppb)');ax.set_title(title);ax.grid(alpha=.2)
fig.colorbar(p,ax=axes.ravel().tolist(),label='L3 gridded support proxy',shrink=.8)
fig.suptitle('Exploratory calendar-month departures (available months only)')
p=FIGURES/'methane11_coverage_sensitive_scatter.png';fig.savefig(p,dpi=230,bbox_inches='tight');plt.show();plt.close(fig)
print('Saved:',p)

## 11G — Save processing summary and L2/native-footprint validation gate

**Stop conditions for emissions work:** Native L2 sounding QA, retrieval counts, pixel-footprint overlap, uncertainty, background atmospheric variability and atmospheric transport have not been established by any analysis here. We must complete those before emission attribution or claimed flux estimates.

In [ ]:
from datetime import datetime, timezone

def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

manifest={
    'notebook':'11_tropomi_methane_qa_2019_2025.ipynb',
    'generated_utc':datetime.now(timezone.utc).isoformat(),
    'source_csv':str(SOURCE.relative_to(ROOT)).replace('\\','/'),
    'source_csv_sha256':sha256(SOURCE),
    'radar_csv':str(RADAR.relative_to(ROOT)).replace('\\','/'),
    'radar_csv_sha256':sha256(RADAR),
    'total_months':int(len(m)),
    'roi_xch4_months':int(m.xch4_available.sum()),
    'roi_minus_ring_months':int(m.difference_available.sum()),
    'data_product':'COPERNICUS/S5P/OFFL/L3_CH4',
    'monthly_xch4_band':'CH4_column_volume_mixing_ratio_dry_air_bias_corrected',
    'l3_grid_coverage_is_native_retrieval_fraction':False,
    'native_l2_sounding_qa_verified':False,
    'native_pixel_footprints_verified':False,
    'retrieval_uncertainty_propagated':False,
    'atmospheric_transport_accounted_for':False,
    'attribution_to_local_methane_flux':False,
    'correlations_causal':False,
    'known_provider_gap':'2022-07-26 through 2022-08-31',
}
path=TABLES/'methane11_qa_manifest.json'
path.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
m.to_csv(TABLES/'methane11_monthly_qa_joined.csv',index=False)

report=f"""# Notebook 11 — TROPOMI XCH₄ observation quality (2019–2025)

- **Status:** Preliminary Level-3 observation audit; native Level-2 retrieval verification outstanding.
- **Months in calendar:** {len(m)}.
- **ROI XCH₄ months:** {int(m.xch4_available.sum())}; unavailable: {int((~m.xch4_available).sum())}.
- **ROI-minus-ring months:** {int(m.difference_available.sum())}.
- **L3 support proxy:** resampled grid fraction, not independent footprints or soundings.
- **Known data gap:** 26 July–31 August 2022 (Earth Engine dataset documentation).

## What cannot be inferred

Neither the Level-3 ROI XCH₄ series nor the ROI-minus-ring contrast is a measurement of Hakaluki wetland methane emissions. Retrieval availability depends on viewing/atmospheric conditions and can be systematically biased by season. Do not fill missing months for causal or flux estimation. Differences across years can reflect background atmospheric methane growth, instrument processing versions and nonlocal transport.

## Scientific requirements before methane attribution

1. Retrieve *native Level-2* methane granules; audit sounding `qa_value`, coverage, footprint intersection, per-sounding precision and supplementary retrieval flags.
2. Check reference region, land/water retrieval bias and monsoon availability, including the 2022 provider gap.
3. Check processing version consistency and bias correction; use averaging kernels/vertical sensitivity if atmospheric modelling.
4. Use meteorological winds and a defensible background or transport/inversion model. Obtain ground flux observations for local validation when feasible.
5. Reevaluate temporal associations with retrieval-count/precision weighting and seasonal missingness-aware sensitivity analyses.

## References

- https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_OFFL_L3_CH4
- https://www.tropomi.eu/data-products/methane
"""
report_path=REPORTS/'NOTEBOOK11_TROPOMI_QA_FINDINGS.md'
report_path.write_text(report,encoding='utf-8')
print('=== 11G SAVED ===')
print('Manifest:',path)
print('Report:',report_path)
print('Monthly audit:',TABLES/'methane11_monthly_qa_joined.csv')
print('NOTE: Native Level-2 methane assessment has NOT been completed.')

## Interpretation rules / next study

**Do not automatically proceed to local wetland methane flux modelling.** First inspect the missingness and the potential selection bias. If May–October methane coverage is poor, any wet-season correlations will be conditional on the rare observed retrievals. Next is a focused **Level-2 native sounding and footprint audit** using ESA/Copernicus data, meteorological winds and an explicit attribution framework. Statistical regression of ROI-averaged Level-3 methane against inundation alone is not a flux model.